# LeetCode 2048 — Next Greater Numerically Balanced Number bằng Python

Link đề: https://leetcode.com/problems/next-greater-numerically-balanced-number/ · Độ khó: **Medium** · Chủ đề: Hash Table, Math, Backtracking, Counting, Enumeration

Skeleton LeetCode:

```python
class Solution:
    def nextBeautifulNumber(self, n: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Hiểu định nghĩa "**cân bằng số**": mỗi chữ số `d` có mặt thì xuất hiện **đúng** `d` lần ⇒ chữ số `0` không bao giờ xuất hiện, và **tổng các chữ số khác nhau** = **số chữ số**.
2. Ước lượng không gian lời giải: với `n ≤ 10⁶`, đáp án có tối đa **7 chữ số** (`1224444`), nên **duyệt tăng dần** từ `n + 1` đã đủ nhanh.
3. **Sinh toàn bộ** số cân bằng bằng cách chọn **tập chữ số** có tổng = độ dài rồi **hoán vị không trùng**; sau đó `bisect`.
4. Viết **backtracking** sinh số theo thứ tự tăng dần với ràng buộc đếm.
5. Tránh bẫy: "**lớn hơn hẳn**" `n`, chữ số `0`, và đáp án có thể **dài hơn** `n`.

## 2. Đề bài & ví dụ

Số nguyên `x` là **cân bằng số** (numerically balanced) nếu với mỗi chữ số `d` xuất hiện trong `x`, có **đúng** `d` lần xuất hiện của `d` trong `x`.

Cho số nguyên `n`, trả về số cân bằng **nhỏ nhất lớn hơn hẳn** `n`.

**Ví dụ 1:** `n = 1` → `22` (chữ số 2 xuất hiện 2 lần).

**Ví dụ 2:** `n = 1000` → `1333` (1 xuất hiện 1 lần, 3 xuất hiện 3 lần). `1022` không hợp lệ vì `0` xuất hiện (nhiều hơn 0 lần).

**Ví dụ 3:** `n = 3000` → `3133`.

**Ràng buộc:** `0 ≤ n ≤ 10⁶`.

In [1]:
import random
import time
import bisect
from collections import Counter
from itertools import combinations, permutations
from typing import List

examples = [(1, 22), (1000, 1333), (3000, 3133)]
for n, exp in examples:
    print(f"n={n} -> kỳ vọng {exp}")

n=1 -> kỳ vọng 22
n=1000 -> kỳ vọng 1333
n=3000 -> kỳ vọng 3133


## 3. Kiến thức nền

### 3.1 Đếm chữ số

`Counter(str(x))` cho tần suất từng chữ số. Kiểm tra cân bằng: `all(int(d) == c for d, c in Counter(str(x)).items())`. Chữ số `'0'` với tần suất `c ≥ 1` luôn vi phạm (`0 ≠ c`).

In [2]:
def is_balanced(x):
    return all(int(d) == c for d, c in Counter(str(x)).items())

for x in (1, 22, 122, 1022, 1333, 3133, 4444, 4441):
    print(x, Counter(str(x)), "->", is_balanced(x))

1 Counter({'1': 1}) -> True
22 Counter({'2': 2}) -> True
122 Counter({'2': 2, '1': 1}) -> True
1022 Counter({'2': 2, '1': 1, '0': 1}) -> False
1333 Counter({'3': 3, '1': 1}) -> True
3133 Counter({'3': 3, '1': 1}) -> True
4444 Counter({'4': 4}) -> True
4441 Counter({'4': 3, '1': 1}) -> False


### 3.2 Cấu trúc của số cân bằng

Nếu tập chữ số khác nhau là `S`, thì độ dài số là `Σ_{d∈S} d`. Ví dụ `S = {1, 3}` ⇒ độ dài 4 (`1333`, `3133`, `3313`, `3331`). Số cân bằng độ dài `L` ứng với các tập con `S ⊆ {1..9}` có tổng `L`, mỗi tập cho `L! / Π d!` hoán vị khác nhau.

### 3.3 Hoán vị không trùng

`itertools.permutations` sinh **cả** các hoán vị trùng nhau (khi có chữ số lặp). Dùng `set(...)` để loại trùng, hoặc backtracking theo bảng đếm.

In [3]:
p = list(permutations("1333"))
print("permutations sinh", len(p), "bộ, khác nhau:", sorted(set("".join(x) for x in p)))

permutations sinh 24 bộ, khác nhau: ['1333', '3133', '3313', '3331']


### 3.4 `bisect_right` cho "phần tử nhỏ nhất lớn hơn hẳn"

Trên list đã sắp `A`, `A[bisect_right(A, n)]` là phần tử nhỏ nhất `> n`.

In [4]:
A = [1, 22, 122, 212, 221, 333]
for n in (0, 1, 22, 100, 300):
    print(f"n={n}: phần tử nhỏ nhất > n là {A[bisect.bisect_right(A, n)]}")

n=0: phần tử nhỏ nhất > n là 1
n=1: phần tử nhỏ nhất > n là 22
n=22: phần tử nhỏ nhất > n là 122
n=100: phần tử nhỏ nhất > n là 122
n=300: phần tử nhỏ nhất > n là 333


## 4. Ý tưởng nền tảng / trực giác

Liệt kê số cân bằng theo độ dài:

| độ dài | tập chữ số | các số |
|---|---|---|
| 1 | {1} | 1 |
| 2 | {2} | 22 |
| 3 | {1,2}, {3} | 122, 212, 221, 333 |
| 4 | {1,3}, {4} | 1333, 3133, 3313, 3331, 4444 |
| 5 | {1,4}, {2,3}, {5} | 14444, …, 22333, …, 55555 |
| 6 | {1,2,3}, {1,5}, {2,4}, {6} | 122333, … |
| 7 | {1,2,4}, {1,6}, {2,5}, {3,4}, {7} | **1224444**, … |

Với `n ≤ 10⁶` (tối đa 7 chữ số, và `10⁶` chỉ có 7 chữ số khi bằng đúng `10⁶`), đáp án lớn nhất là số cân bằng 7 chữ số nhỏ nhất `1224444`. Khoảng cách từ `n` tới đáp án tối đa vài trăm nghìn ⇒ **duyệt từng số** cũng được; **sinh sẵn** thì nhanh hơn nhiều.

Tính tay **Ví dụ 2** (`n = 1000`): các số 4 chữ số cân bằng theo thứ tự `1333, 3133, 3313, 3331, 4444` ⇒ số đầu tiên `> 1000` là `1333`. **Ví dụ 3** (`n = 3000`): `1333 < 3000` ⇒ `3133`.

In [5]:
four = sorted(int("".join(p)) for S in ({1, 3}, {4}) for p in set(permutations("".join(str(d) * d for d in sorted(S)))))
print("số cân bằng 4 chữ số:", four)

số cân bằng 4 chữ số: [1333, 3133, 3313, 3331, 4444]


## 5. Cách 1 — Brute Force / baseline: thử `n + 1, n + 2, …`

In [6]:
def next_bruteforce(n):
    x = n + 1
    while not is_balanced(x):
        x += 1
    return x

for n, exp in examples:
    got = next_bruteforce(n); print(n, "->", got, "kỳ vọng", exp); assert got == exp

1 -> 22 kỳ vọng 22
1000 -> 1333 kỳ vọng 1333
3000 -> 3133 kỳ vọng 3133


**Độ phức tạp:** số lần thử = `đáp án − n`, tệ nhất khi `n` vừa vượt số cân bằng 6 chữ số lớn nhất `666666`: phải thử tới `1224444` ⇒ `~5.6·10⁵` lần, mỗi lần `O(7)`. Khoảng vài trăm ms trong Python — vẫn AC nhưng không đẹp.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (đếm khả năng):** số cân bằng rất **thưa**: chỉ có vài trăm số có tối đa 7 chữ số. Sinh hết một lần rồi tra cứu.

**Insight 2 (sinh theo tập chữ số):** chọn tập `S ⊆ {1..7}` với `Σ S ≤ 7`; dựng chuỗi "mẫu" có `d` chữ số `d` với mỗi `d ∈ S`; lấy mọi hoán vị không trùng.

**Insight 3 (backtracking có thứ tự):** với độ dài `L` cố định, điền từng vị trí từ trái sang phải, thử chữ số tăng dần, với bảng "còn lại" `rem[d]`; chỉ chọn các tập `S` có tổng `L`. Thứ tự sinh là tăng dần ⇒ số đầu tiên `> n` là đáp án.

**Cận trên của độ dài:** `n ≤ 10⁶` ⇒ đáp án `≤ 1224444` (7 chữ số); sinh tới độ dài 7 là đủ.

### 6.1 Đếm số cân bằng theo độ dài

Đếm bằng công thức đa thức `L! / Π d!` cho mỗi tập và so với số lượng sinh thật.

In [7]:
from math import factorial
def count_by_formula(L):
    total = 0
    for r in range(1, 10):
        for S in combinations(range(1, 10), r):
            if sum(S) == L:
                c = factorial(L)
                for d in S: c //= factorial(d)
                total += c
    return total

for L in range(1, 9):
    print(f"độ dài {L}: {count_by_formula(L)} số cân bằng")

độ dài 1: 1 số cân bằng
độ dài 2: 1 số cân bằng
độ dài 3: 4 số cân bằng
độ dài 4: 5 số cân bằng
độ dài 5: 16 số cân bằng
độ dài 6: 82 số cân bằng
độ dài 7: 169 số cân bằng
độ dài 8: 541 số cân bằng


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sinh toàn bộ (tới 7 chữ số) + `bisect` (bản nộp)

In [8]:
def gen_balanced(max_len=7):
    out = set()
    for r in range(1, 8):
        for S in combinations(range(1, 8), r):
            if sum(S) <= max_len:
                pattern = "".join(str(d) * d for d in S)
                out.update(int("".join(p)) for p in permutations(pattern))
    return sorted(out)

BAL = gen_balanced()
print(len(BAL), "số cân bằng ≤ 7 chữ số; đầu:", BAL[:12], "... cuối:", BAL[-3:])

def next_gen(n):
    return BAL[bisect.bisect_right(BAL, n)]

print([next_gen(n) for n, _ in examples])

278 số cân bằng ≤ 7 chữ số; đầu: [1, 22, 122, 212, 221, 333, 1333, 3133, 3313, 3331, 4444, 14444] ... cuối: [6666616, 6666661, 7777777]
[22, 1333, 3133]


Tập `{1, 2, 4}` có tổng 7 sinh `7!/(1!·2!·4!) = 105` số; `permutations` sinh `5040` bộ cho mỗi mẫu 7 ký tự rồi loại trùng — vẫn rất nhanh. Tổng số mẫu ít nên tiền xử lý chỉ vài chục ms.

**Độ phức tạp Cách 2:** tiền xử lý `O(Σ L!)` với `L ≤ 7` (hằng số), mỗi truy vấn `O(log K)` với `K` vài trăm.

### 7.2 Cách 3 — Backtracking theo độ dài, sinh tăng dần, dừng sớm

In [9]:
def next_backtrack(n):
    L0 = len(str(n))
    for L in range(L0, 9):
        best = None
        for r in range(1, 10):
            for S in combinations(range(1, 10), r):
                if sum(S) != L:
                    continue
                rem = {d: d for d in S}
                digits = []
                def dfs():
                    nonlocal best
                    if len(digits) == L:
                        x = int("".join(map(str, digits)))
                        if x > n:
                            best = x if best is None else min(best, x)
                            return True             # sinh tăng dần: số đầu tiên > n là nhỏ nhất của tập S
                        return False
                    for d in sorted(rem):
                        if rem[d]:
                            rem[d] -= 1; digits.append(d)
                            if dfs(): rem[d] += 1; digits.pop(); return True
                            rem[d] += 1; digits.pop()
                    return False
                dfs()
        if best is not None:
            return best
    return None

print([next_backtrack(n) for n, _ in examples])

[22, 1333, 3133]


Với mỗi độ dài `L` (bắt đầu từ số chữ số của `n`), mỗi tập `S` có tổng `L` cho số nhỏ nhất `> n` của tập đó (DFS sinh tăng dần, dừng ngay khi gặp số đầu tiên `> n`). Lấy `min` qua các tập; nếu độ dài `L` không có ứng viên thì thử `L + 1`. Cách này mở rộng được cho `n` lớn hơn nhiều (Bài tập 5).

**Độ phức tạp Cách 3:** phụ thuộc số tập và độ sâu cắt tỉa; với `L ≤ 7` rất nhỏ.

### 7.3 Cách 4 — Duyệt tăng dần với kiểm tra nhanh bằng mảng đếm

Cải tiến hằng số của Cách 1: bỏ `Counter`/`str`, đếm chữ số bằng phép chia và **loại sớm** khi gặp `0` hoặc khi một chữ số vượt quá chính nó.

In [10]:
def is_balanced_fast(x):
    cnt = [0] * 10
    while x:
        x, d = divmod(x, 10)
        if d == 0:
            return False
        cnt[d] += 1
        if cnt[d] > d:
            return False
    return all(c == 0 or c == d for d, c in enumerate(cnt))

def next_fast_scan(n):
    x = n + 1
    while not is_balanced_fast(x):
        x += 1
    return x

print([next_fast_scan(n) for n, _ in examples])

[22, 1333, 3133]


### 7.4 Khoảng trống lớn nhất giữa hai số cân bằng liên tiếp

Chi phí của Cách 1 bằng khoảng cách từ `n` tới đáp án, tức tối đa là **khoảng trống lớn nhất** giữa hai số cân bằng liên tiếp (trong vùng `n ≤ 10⁶`). Tính trực tiếp từ danh sách sinh sẵn — đây là cách định lượng "duyệt tuyến tính có đủ nhanh không" thay vì đoán.

In [11]:
gaps = sorted(((b - a, a, b) for a, b in zip(BAL, BAL[1:]) if a <= 10 ** 6), reverse=True)[:5]
for g, a, b in gaps:
    print(f"khoảng trống {g:>7}: từ {a} tới {b}")
print("=> Cách 1 thử tối đa khoảng", gaps[0][0], "số cho một truy vấn")

khoảng trống  557778: từ 666666 tới 1224444
khoảng trống  111115: từ 555551 tới 666666
khoảng trống   89223: từ 333221 tới 422444
khoảng trống   71133: từ 444422 tới 515555
khoảng trống   67791: từ 244442 tới 312233
=> Cách 1 thử tối đa khoảng 557778 số cho một truy vấn


## 8. TRACE từng bước

Trace Cách 3 cho **Ví dụ 3** (`n = 3000`): độ dài 4, các tập `{1,3}` và `{4}`.

In [12]:
def trace_backtrack(n):
    L = len(str(n))
    for S in [S for r in range(1, 10) for S in combinations(range(1, 10), r) if sum(S) == L]:
        pattern = "".join(str(d) * d for d in S)
        cands = sorted(set(int("".join(p)) for p in permutations(pattern)))
        first = next((x for x in cands if x > n), None)
        print(f"tập {set(S)}: các số {cands} -> nhỏ nhất > {n}: {first}")

trace_backtrack(3000)
print("=> đáp án min(3133, 4444) = 3133")

tập {4}: các số [4444] -> nhỏ nhất > 3000: 4444
tập {1, 3}: các số [1333, 3133, 3313, 3331] -> nhỏ nhất > 3000: 3133
=> đáp án min(3133, 4444) = 3133


| tập | các số (tăng dần) | nhỏ nhất `> 3000` |
|---|---|---|
| {1,3} | 1333, 3133, 3313, 3331 | **3133** |
| {4} | 4444 | 4444 |

Trace thêm `n = 666666` — không có số 6 chữ số nào lớn hơn (số cân bằng 6 chữ số lớn nhất chính là `666666`), nên phải sang độ dài 7:

In [13]:
six = [x for x in BAL if len(str(x)) == 6]
print("số cân bằng 6 chữ số lớn nhất:", max(six), "| next(666666) =", next_gen(666666), "=", next_backtrack(666666))

số cân bằng 6 chữ số lớn nhất: 666666 | next(666666) = 1224444 = 1224444


## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 2** — sinh toàn bộ một lần (ở mức lớp) rồi `bisect`.

In [14]:
class Solution:
    BAL = None

    def nextBeautifulNumber(self, n: int) -> int:
        if Solution.BAL is None:
            out = set()
            for r in range(1, 8):
                for S in combinations(range(1, 8), r):
                    if sum(S) <= 7:
                        pattern = "".join(str(d) * d for d in S)
                        out.update(int("".join(p)) for p in permutations(pattern))
            Solution.BAL = sorted(out)
        return Solution.BAL[bisect.bisect_right(Solution.BAL, n)]

for n, exp in examples:
    got = Solution().nextBeautifulNumber(n); print(got, exp); assert got == exp

22 22
1333 1333
3133 3133


Bảng `BAL` lưu ở **thuộc tính lớp** nên chỉ dựng một lần dù LeetCode tạo nhiều đối tượng `Solution`.

## 10. Vì sao thuật toán đúng

**Đặc trưng số cân bằng.** Nếu `x` cân bằng và `S` là tập chữ số xuất hiện, thì `0 ∉ S` (vì `0` phải xuất hiện 0 lần), mỗi `d ∈ S` xuất hiện đúng `d` lần ⇒ `x` là một hoán vị của mẫu `"d…d"` (ghép các khối `d` lặp `d` lần), độ dài `Σ S`. Ngược lại mọi hoán vị như vậy đều cân bằng. Vậy tập số cân bằng có `≤ 7` chữ số = hợp các tập hoán vị của mẫu ứng với `S ⊆ {1..7}`, `Σ S ≤ 7` (chữ số `8`, `9` cần ít nhất 8, 9 chữ số) — đúng như `gen_balanced`.

**Đủ 7 chữ số.** `1224444` cân bằng và `> 10⁶ ≥ n` ⇒ đáp án `≤ 1224444` luôn có trong `BAL`.

**`bisect_right`.** `BAL` sắp tăng; `bisect_right(BAL, n)` là chỉ số đầu tiên có `BAL[i] > n` ⇒ đúng "nhỏ nhất lớn hơn hẳn". ∎

### 10.1 Kiểm chứng thực nghiệm: `BAL` = mọi số cân bằng `≤ 1 300 000`

Quét toàn bộ `[1, 1 300 000]` bằng `is_balanced_fast` và so với danh sách sinh ra (lọc phần `≤ 1 300 000`).

In [15]:
t = time.perf_counter()
scan = [x for x in range(1, 1_300_001) if is_balanced_fast(x)]
print(f"quét: {len(scan)} số ({(time.perf_counter() - t):.2f} s)")
assert scan == [x for x in BAL if x <= 1_300_000]
print("Danh sách sinh = quét thật trên [1, 1.3e6]; cũng khớp công thức đếm:", [count_by_formula(L) for L in range(1, 8)],
      [sum(1 for x in BAL if len(str(x)) == L) for L in range(1, 8)])

quét: 114 số (0.59 s)
Danh sách sinh = quét thật trên [1, 1.3e6]; cũng khớp công thức đếm: [1, 1, 4, 5, 16, 82, 169] [1, 1, 4, 5, 16, 82, 169]


### 10.2 Kiểm chứng ba cách trên một đoạn liên tục của `n`

Với mọi `n` trong vài đoạn liên tục (quanh các số cân bằng và quanh mốc đổi độ dài), Cách 2, Cách 3 và quét nhanh cho cùng kết quả. Các đoạn này chứa đúng những chỗ dễ sai: `n` bằng một số cân bằng, `n` ngay trước/sau nó, và `n` lớn hơn số cân bằng lớn nhất của một độ dài.

In [16]:
ranges = [range(0, 400), range(1300, 1400), range(3300, 3340), range(4440, 4450), range(55550, 55560), range(666660, 666670)]
cnt = 0
for rg in ranges:
    for n in rg:
        a = next_gen(n)
        assert a == next_backtrack(n) == next_fast_scan(n), n
        cnt += 1
print(f"Cách 2 = Cách 3 = quét nhanh trên {cnt} giá trị n liên tục")

Cách 2 = Cách 3 = quét nhanh trên 570 giá trị n liên tục


## 11. Phân tích Time / Space Complexity

| Cách | Tiền xử lý | Mỗi truy vấn | Space |
|---|---|---|---|
| 1. Duyệt + `Counter` | — | `O((ans − n) · 7)` ≈ `5.6·10⁵` tệ nhất | `O(1)` |
| 2. Sinh + `bisect` | `O(Σ L!)`, hằng | `O(log K)` | `O(K)` |
| 3. Backtracking theo độ dài | — | nhỏ (cắt tỉa) | `O(L)` |
| 4. Duyệt + đếm nhanh | — | như 1, hằng số nhỏ hơn | `O(1)` |

In [17]:
worst = [666666, 10 ** 6, 0, 555555]
for name, f in [("brute", next_bruteforce), ("fast_scan", next_fast_scan), ("backtrack", next_backtrack), ("submit", Solution().nextBeautifulNumber)]:
    t = time.perf_counter(); r = [f(n) for n in worst]; print(f"{name:<10}: {r} ({(time.perf_counter() - t) * 1000:.0f} ms)")

brute     : [1224444, 1224444, 1, 666666] (794 ms)


fast_scan : [1224444, 1224444, 1, 666666] (368 ms)
backtrack : [1224444, 1224444, 1, 666666] (0 ms)
submit    : [1224444, 1224444, 1, 666666] (0 ms)


## 12. Bộ test

In [18]:
ALL = {"gen": next_gen, "backtrack": next_backtrack, "fast_scan": next_fast_scan, "submit": Solution().nextBeautifulNumber}
def run_all(n, brute=True):
    res = {k: f(n) for k, f in ALL.items()}
    if brute:
        res["brute"] = next_bruteforce(n)
    assert len(set(res.values())) == 1, (n, res)
    return res["submit"]

for n, exp in examples:
    assert run_all(n) == exp
# ví dụ tự dựng (đối chiếu lại đề gốc) — kiểm bằng vét cạn trong run_all
edge = [(0, 1), (22, 122), (21, 22), (333, 1333), (1333, 3133), (3331, 4444), (4444, 14444),
        (666666, 1224444), (10 ** 6, 1224444), (122, 212)]
for n, exp in edge:
    assert run_all(n) == exp, (n, run_all(n))
rng = random.Random(20480)
for _ in range(300):
    run_all(rng.randint(0, 10 ** 6), brute=False)
for _ in range(20):
    run_all(rng.randint(0, 2 * 10 ** 5))                 # vét cạn chỉ với n nhỏ để giữ thời gian chạy
print("Ví dụ, 10 edge case, 320 n ngẫu nhiên: OK")

Ví dụ, 10 edge case, 320 n ngẫu nhiên: OK


### 12.1 Kiểm tra tính chất

- Đáp án luôn cân bằng và `> n`.
- Không có số cân bằng nào trong `(n, đáp án)`.
- Hàm không giảm theo `n`; và `f(f(n) − 1) = f(n)`.

In [19]:
rng = random.Random(121)
f = Solution().nextBeautifulNumber
prev = 0
for n in sorted(rng.randint(0, 10 ** 6) for _ in range(2000)):
    a = f(n)
    assert a > n and is_balanced_fast(a)
    i = bisect.bisect_right(BAL, n)
    assert BAL[i] == a and (i == 0 or BAL[i - 1] <= n)
    assert a >= prev and f(a - 1) == a
    prev = a
print("4 tính chất đúng trên 2000 giá trị n")

4 tính chất đúng trên 2000 giá trị n


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `>=` (trả về chính `n` nếu `n` cân bằng)

In [20]:
def wrong_ge(n):
    return BAL[bisect.bisect_left(BAL, n)]                     # SAI: bisect_left cho phần tử >= n

for n in (1, 22, 1333):
    print(f"n={n}: sai {wrong_ge(n)} | đúng {Solution().nextBeautifulNumber(n)}")

n=1: sai 1 | đúng 22
n=22: sai 22 | đúng 122
n=1333: sai 1333 | đúng 3133


**Vì sao sai:** đề yêu cầu **lớn hơn hẳn**. Ví dụ 1: `n = 1` là số cân bằng nhưng đáp án là `22`. **Sửa:** `bisect_right` (hoặc bắt đầu duyệt từ `n + 1`).

### Lỗi 2 — Bỏ qua chữ số `0` khi kiểm tra

In [21]:
def wrong_skip_zero(n):
    x = n + 1
    while not all(int(d) == c for d, c in Counter(str(x)).items() if d != "0"):   # SAI: bỏ qua '0'
        x += 1
    return x

for n, exp in examples:
    print(f"n={n}: sai {wrong_skip_zero(n)} | đúng {exp}")

n=1: sai 10 | đúng 22
n=1000: sai 1022 | đúng 1333
n=3000: sai 3033 | đúng 3133


**Vì sao sai:** `0` phải xuất hiện đúng 0 lần; bỏ qua nó khiến `10` ("1 xuất hiện 1 lần") hay `1022` được chấp nhận. Đề nhấn mạnh điều này ở Ví dụ 2. **Sửa:** không loại `'0'` (điều kiện `int('0') == c` tự động sai).

### Lỗi 3 — Kiểm tra `count ≥ d` thay vì `== d`

In [22]:
def wrong_ge_count(n):
    x = n + 1
    while not all(c >= int(d) and d != "0" for d, c in Counter(str(x)).items()):    # SAI
        x += 1
    return x

for n in (1, 22, 1000):
    print(f"n={n}: sai {wrong_ge_count(n)} | đúng {Solution().nextBeautifulNumber(n)}")

n=1: sai 11 | đúng 22
n=22: sai 111 | đúng 122
n=1000: sai 1111 | đúng 1333


**Vì sao sai:** `11` có chữ số 1 xuất hiện 2 lần (`≥ 1`) nên bị nhận nhầm. **Sửa:** so sánh **bằng**.

### Lỗi 4 — Chỉ sinh các số có chữ số sắp tăng (quên hoán vị)

In [23]:
def gen_sorted_only():
    out = []
    for r in range(1, 8):
        for S in combinations(range(1, 8), r):
            if sum(S) <= 7:
                out.append(int("".join(str(d) * d for d in S)))   # SAI: chỉ một cách sắp
    return sorted(out)

BAD = gen_sorted_only()
for n in (1333, 3000, 122):
    print(f"n={n}: sai {BAD[bisect.bisect_right(BAD, n)]} | đúng {Solution().nextBeautifulNumber(n)}")

n=1333: sai 4444 | đúng 3133
n=3000: sai 4444 | đúng 3133
n=122: sai 333 | đúng 212


**Vì sao sai:** `3133`, `212`, `221`… cũng cân bằng — mọi **hoán vị** của mẫu đều hợp lệ. **Sửa:** lấy mọi hoán vị (loại trùng).

### Lỗi 5 — Chỉ tìm trong cùng số chữ số với `n`

In [24]:
def wrong_same_length(n):
    L = len(str(n))
    cands = [x for x in BAL if len(str(x)) == L and x > n]       # SAI: có thể rỗng
    return min(cands) if cands else None

for n in (333, 4444, 666666):
    print(f"n={n}: sai {wrong_same_length(n)} | đúng {Solution().nextBeautifulNumber(n)}")

n=333: sai None | đúng 1333
n=4444: sai None | đúng 14444
n=666666: sai None | đúng 1224444


**Vì sao sai:** `n = 4444` là số cân bằng 4 chữ số lớn nhất ⇒ đáp án `14444` có 5 chữ số. **Sửa:** nếu độ dài hiện tại không có ứng viên, chuyển sang độ dài kế tiếp (Cách 3) hoặc sinh mọi độ dài (Cách 2).

### Lỗi 6 — Chỉ sinh tới 6 chữ số

In [25]:
BAL6 = gen_balanced(6)
def wrong_six(n):
    i = bisect.bisect_right(BAL6, n)
    return BAL6[i] if i < len(BAL6) else "IndexError"

for n in (666665, 666666, 10 ** 6):
    print(f"n={n}: sai {wrong_six(n)} | đúng {Solution().nextBeautifulNumber(n)}")

n=666665: sai 666666 | đúng 666666
n=666666: sai IndexError | đúng 1224444
n=1000000: sai IndexError | đúng 1224444


**Vì sao sai:** `n` có thể tới `10⁶` (và mọi `n ≥ 666666`) cần đáp án 7 chữ số `1224444`. **Sửa:** sinh tới độ dài 7 (chứng minh ở §10).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Tập lời giải **rất thưa** và có cấu trúc ⇒ **sinh sẵn** (precompute) rồi tra cứu (`bisect`).
- "Mỗi chữ số `d` xuất hiện `d` lần" ⇒ độ dài = tổng tập chữ số ⇒ liệt kê theo **tập con có tổng cố định**.
- Hoán vị có lặp: `set(permutations(...))` với chuỗi ngắn; backtracking với bảng đếm khi dài.
- Ước lượng cận đáp án trước khi chọn duyệt tuyến tính (`ans − n` bao nhiêu?).
- "Lớn hơn hẳn" ⇒ `bisect_right` / bắt đầu từ `n + 1`.

## 15. Biến thể của bài

1. **Đếm số cân bằng trong `[lo, hi]`** — hai lần `bisect` (Bài tập 4).
2. **`n` tới `10¹⁵`** — sinh theo độ dài với backtracking có cận (Bài tập 5).
3. **31 Next Permutation / 556 Next Greater Element III**: số nhỏ nhất lớn hơn với **cùng** tập chữ số.
4. **Định nghĩa khác**: chữ số `d` xuất hiện **ít nhất** `d` lần, hoặc đúng `d + 1` lần — đổi mẫu sinh.
5. **902 Numbers At Most N Given Digit Set**: đếm số tạo từ tập chữ số cho trước — digit DP.

### 15.1 Code biến thể: 556 Next Greater Element III (next permutation)

In [26]:
def next_greater_element_iii(n):
    d = list(str(n)); i = len(d) - 2
    while i >= 0 and d[i] >= d[i + 1]:
        i -= 1
    if i < 0:
        return -1
    j = len(d) - 1
    while d[j] <= d[i]:
        j -= 1
    d[i], d[j] = d[j], d[i]
    d[i + 1:] = reversed(d[i + 1:])
    x = int("".join(d))
    return x if x < 2 ** 31 else -1

def ng3_brute(n):
    s = str(n); c = sorted(int("".join(p)) for p in set(permutations(s)))
    nxt = [x for x in c if x > n]
    return nxt[0] if nxt and nxt[0] < 2 ** 31 else -1

rng = random.Random(556)
for _ in range(300):
    n = rng.randint(1, 10 ** 6); assert next_greater_element_iii(n) == ng3_brute(n)
print("556 khớp vét cạn;", next_greater_element_iii(1333), "(hoán vị kế tiếp của 1333 cũng là 3133)")

556 khớp vét cạn; 3133 (hoán vị kế tiếp của 1333 cũng là 3133)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Truy vấn | Khi nào dùng |
|---|---|---|---|
| 1. Duyệt + `Counter` | Thử từng số | `~10⁵` lần thử | Kiểm chứng |
| 2. Sinh + `bisect` | Precompute vài trăm số | `O(log K)` | **Bản nộp** |
| 3. Backtracking theo độ dài | Sinh tăng dần, dừng sớm | nhỏ | `n` rất lớn |
| 4. Duyệt + đếm nhanh | Loại sớm | như 1 | Code ngắn, đủ nhanh |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `n = 0`; (b) `n = 22`; (c) `n = 333`; (d) `n = 4444`.

**Bài 2 (tính tay):** Liệt kê mọi số cân bằng có 5 chữ số bắt đầu bằng chữ số `2`, và cho biết `next(22000)`.

**Bài 3 (điền TODO):** Hoàn thiện `is_balanced_todo(x)` không dùng `Counter`.

**Bài 4 (tự viết lại):** Viết `count_balanced(lo, hi)` — số số cân bằng trong `[lo, hi]` (với `hi ≤ 10⁷`).

**Bài 5 (mini challenge):** Viết `next_balanced_big(n)` cho `n < 10¹⁰` (đáp án có thể tới 11 chữ số) bằng cách sinh theo độ dài với backtracking có thứ tự; đối chiếu `Solution` với `n ≤ 10⁶` và kiểm tra kết quả cân bằng với `n` lớn.

In [27]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def is_balanced_todo(x):
    cnt = [0] * 10
    # TODO: tách từng chữ số bằng divmod, tăng cnt; cuối cùng kiểm tra cnt[d] in (0, d) và cnt[0] == 0
    pass

In [28]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def count_balanced(lo, hi):
    # TODO: sinh tới 8 chữ số; bisect_right(hi) - bisect_left(lo)
    pass

def next_balanced_big(n):
    # TODO: với L = len(str(n)), L+1, ...: với mỗi tập S (tổng L) DFS sinh tăng dần, có cắt tỉa theo tiền tố của n
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: đọc bảng mục 4. Mức 2: "lớn hơn hẳn". Mức 3: (a) `1`; (b) `122`; (c) `1333`; (d) `14444`.

**Bài 2.** Mức 1: tập có tổng 5: `{1,4}`, `{2,3}`, `{5}`. Mức 2: bắt đầu bằng 2 ⇒ tập `{2,3}`: hoán vị của `22333` bắt đầu bằng `2`: `22333, 23233, 23323, 23332`. Mức 3: `next(22000) = 22333`.

**Bài 3.** Mức 1: `while x: x, d = divmod(x, 10)`. Mức 2: `cnt[0]` phải là 0. Mức 3: `all(c in (0, d) for d, c in enumerate(cnt))`.

**Bài 4.** Mức 1: sinh như Cách 2 với độ dài tới 8 (`10⁷` có 8 chữ số). Mức 2: chữ số 8 cần đúng 8 lần. Mức 3: hai `bisect`.

**Bài 5.** Mức 1: độ dài `L` có các tập `S ⊆ {1..9}`, `Σ S = L`. Mức 2: DFS điền từ trái, nếu tiền tố hiện tại **bằng** tiền tố của `n` thì chỉ được chọn chữ số `≥` chữ số tương ứng của `n`; nếu đã **lớn hơn** thì điền phần còn lại nhỏ nhất. Mức 3: số đầu tiên `> n` tìm được của mỗi tập là nhỏ nhất của tập đó; lấy `min`, nếu không có thì `L + 1`.

## 19. Đáp án tham khảo cho bài tập

In [29]:
# Bài 1, 2, 3
f = Solution().nextBeautifulNumber
b1 = [f(n) for n in (0, 22, 333, 4444)]
print("Bài 1:", b1); assert b1 == [1, 122, 1333, 14444]
b2 = [x for x in BAL if len(str(x)) == 5 and str(x)[0] == "2"]
print("Bài 2:", b2, "| next(22000) =", f(22000)); assert b2 == [22333, 23233, 23323, 23332] and f(22000) == 22333
def is_balanced_todo(x):
    cnt = [0] * 10
    while x:
        x, d = divmod(x, 10); cnt[d] += 1
    return cnt[0] == 0 and all(c in (0, d) for d, c in enumerate(cnt))
assert all(is_balanced_todo(x) == is_balanced(x) for x in range(1, 200000))
print("Bài 3: OK")

Bài 1: [1, 122, 1333, 14444]
Bài 2: [22333, 23233, 23323, 23332] | next(22000) = 22333


Bài 3: OK


In [30]:
# Bài 4, 5
BAL8 = None
def count_balanced(lo, hi):
    global BAL8
    if BAL8 is None:
        out = set()
        for r in range(1, 9):
            for S in combinations(range(1, 9), r):
                if sum(S) <= 8:
                    out.update(int("".join(p)) for p in set(permutations("".join(str(d) * d for d in S))))
        BAL8 = sorted(out)
    return bisect.bisect_right(BAL8, hi) - bisect.bisect_left(BAL8, lo)

def next_balanced_big(n):
    s = str(n)
    for L in range(len(s), 20):
        target = s if L == len(s) else None
        best = None
        for r in range(1, 10):
            for S in combinations(range(1, 10), r):
                if sum(S) != L:
                    continue
                rem = {d: d for d in S}; digits = []
                def dfs(tight):
                    if len(digits) == L:
                        return not tight                   # đã lớn hơn hẳn n
                    i = len(digits)
                    lo = int(target[i]) if (tight and target) else 0
                    for d in sorted(rem):
                        if rem[d] and d >= lo:
                            rem[d] -= 1; digits.append(d)
                            if dfs(tight and target is not None and d == lo):
                                return True
                            rem[d] += 1; digits.pop()
                    return False
                if dfs(target is not None):
                    x = int("".join(map(str, digits)))
                    best = x if best is None else min(best, x)
        if best is not None:
            return best

rng = random.Random(5)
for _ in range(200):
    lo = rng.randint(0, 2 * 10 ** 6); hi = lo + rng.randint(0, 3 * 10 ** 5)
    assert count_balanced(lo, hi) == sum(1 for x in BAL if lo <= x <= hi)
for _ in range(300):
    n = rng.randint(0, 10 ** 6); assert next_balanced_big(n) == f(n)
big = [rng.randint(10 ** 8, 10 ** 10) for _ in range(5)]
res = [next_balanced_big(n) for n in big]
assert all(r > n and is_balanced(r) for r, n in zip(res, big))
print("Bài 4, 5: OK |", count_balanced(1, 10 ** 6), list(zip(big, res))[:2])

Bài 4, 5: OK | 109 [(259352783, 261266666), (900331681, 999999999)]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (sinh sẵn + bisect):**

```python
out = set()
for r in range(1, 8):
    for S in combinations(range(1, 8), r):
        if sum(S) <= 7:
            pat = "".join(str(d) * d for d in S)
            out.update(int("".join(p)) for p in permutations(pat))
BAL = sorted(out)
return BAL[bisect_right(BAL, n)]        # nhỏ nhất > n
```

**Checklist:**
- [ ] "Lớn hơn **hẳn**" (`bisect_right`, bắt đầu từ `n + 1`)?
- [ ] Chữ số `0` không được xuất hiện?
- [ ] Đếm **đúng bằng** `d`?
- [ ] Lấy mọi **hoán vị**, không chỉ một cách sắp?
- [ ] Sinh đủ độ dài (7 chữ số cho `n ≤ 10⁶`)?